# Retail Flow data generation with Faker

## Table of contents

* **1. Setup and Libraries**: Installing and importing required Python packages.
* **2. Static data definitions**: Defining dictionaries for stores, base zones, cameras, and the product catalog.
* **3. Base entities (Stores, Zones, Cameras)**: Multiplying base configurations across all stores to establish Primary and Foreign Keys.
* **4. Sales & Sold Products simulation**: Generating checkout tickets and mapping products to specific store zones.
* **5. Store traffic & bounce rates**: Simulating hourly visitor distribution while strictly respecting store capacity limits.
* **6. Camera detections simulation**: Generating spatial (x, y) client detections bounded within actual camera zones.
* **7. Data export (CSV)**: Saving the generated dataframes to a local directory for analysis.

### 1. Setup and Libraries
First, we will install the required libraries and import the necessary modules. We also initialize Faker to use Spanish for realistic naming.

In [2]:
# %pip install faker pandas

In [13]:
from faker.providers import DynamicProvider
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd
import random
import uuid
import csv

# limit Faker to Spanish
fake = Faker('es_ES')

### 2. Static data definitions
Here we define the static configuration of our retail environment: the stores, the zones inside them, the camera models, and the product catalog.

In [14]:
stores_dict = {
    'SRCL': {
        'store_name': 'Sara Colon',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        },
    'SRGV': {
        'store_name': 'Sara Gran Via',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        },
    'SRAQ': {
        'store_name': 'Sara Aqua',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        },
    'SRBN': {'store_name': 'Sara Bonaire',
             'city': 'Valencia',
             'm2': 585,
             'max_capacity': 290
             },
    'SRSL': {
        'store_name': 'Sara Saler',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        }
}

In [15]:
base_zones = {
    'ENTR': {
        'zone_name': 'Entrance',
        'zone_type': 'Entrance',
        'coord_lims': None
        },
    'WM': {'zone_name': 'Women',
           'zone_type': 'Women clothes',
           'coord_lims': {
               'x_min': 15, 'x_max': 28, 'y_min': 3, 'y_max': 15
               }
           },
    'MN': {
        'zone_name': 'Men',
        'zone_type': 'Men clothes',
        'coord_lims': {
            'x_min': 28, 'x_max': 40, 'y_min': 0, 'y_max': 8
            }
            },
    'KD': {
        'zone_name': 'Kids',
        'zone_type': 'Kid clothes',
        'coord_lims': {
            'x_min': 28, 'x_max': 40, 'y_min': 8, 'y_max': 15
            }
            },
    'CO': {
        'zone_name': 'Checkout',
        'zone_type': 'Checkout',
        'coord_lims': {
            'x_min': 15, 'x_max': 22, 'y_min': 0, 'y_max': 3
            }
            },
    'FR': {
        'zone_name': 'Fitting Rooms',
        'zone_type': 'Fitting Rooms',
        'coord_lims': {
            'x_min': 7, 'x_max': 15, 'y_min': 5, 'y_max': 10
            }
            }
}

In [16]:
base_cameras = {
    'ENTRCM': {
        'zone_id': 'ENTR',
        'model': 'Bullet'
        },
    'WMCM': {
        'zone_id': 'WM',
        'model': 'Fisheye'
        },
    'MNCM': {
        'zone_id': 'MN', 
        'model': 'Fisheye'
        },
    'KDCM': {
        'zone_id': 'KD',
        'model': 'Fisheye'
        },
    'COCM': {
        'zone_id': 'CO',
        'model': 'Turret'
        },
    'FRCM': {
        'zone_id': 'FR',
        'model': 'Dome'
        }
}

In [17]:
# base products w/o a specific ticket_id or zone_id
catalog = [
    {'name': 'Women Shirt', 'category': 'Tops', 'price': 25.99, 'base_zone': 'WM'},
    {'name': 'Women Jeans', 'category': 'Bottoms', 'price': 49.95, 'base_zone': 'WM'},
    {'name': 'Women Summer Dress', 'category': 'Dresses', 'price': 35.50, 'base_zone': 'WM'},
    {'name': 'Women Leather Jacket', 'category': 'Outerwear', 'price': 120.00, 'base_zone': 'WM'},
    {'name': 'Women Wool Sweater', 'category': 'Tops', 'price': 45.00, 'base_zone': 'WM'},

    {'name': 'Men Basic T-Shirt', 'category': 'Tops', 'price': 15.99, 'base_zone': 'MN'},
    {'name': 'Men Chino Trousers', 'category': 'Bottoms', 'price': 39.99, 'base_zone': 'MN'},
    {'name': 'Men Hoodie', 'category': 'Tops', 'price': 45.00, 'base_zone': 'MN'},
    {'name': 'Men Denim Jacket', 'category': 'Outerwear', 'price': 59.95, 'base_zone': 'MN'},
    {'name': 'Men Tailored Suit', 'category': 'Suits', 'price': 250.00, 'base_zone': 'MN'},

    {'name': 'Kids Graphic T-Shirt', 'category': 'Tops', 'price': 12.50, 'base_zone': 'KD'},
    {'name': 'Kids Denim Shorts', 'category': 'Bottoms', 'price': 18.99, 'base_zone': 'KD'},
    {'name': 'Kids Winter Coat', 'category': 'Outerwear', 'price': 45.00, 'base_zone': 'KD'},
    {'name': 'Kids Pajama Set', 'category': 'Nightwear', 'price': 22.50, 'base_zone': 'KD'},
    {'name': 'Kids School Uniform', 'category': 'Sets', 'price': 35.00, 'base_zone': 'KD'}
]

### 3. Base entities (Stores, Zones, Cameras)
We multiply the base zones and cameras across all available stores to generate unique Primary Keys (PK) and maintain Foreign Key (FK) integrity.

In [18]:
# A. STORES
stores_list = [{'store_id': key, **value} for key, value in stores_dict.items()]
df_stores = pd.DataFrame(stores_list)

# B. ZONES & C. CAMERAS
zones_list = []
cameras_list = []

for store_id in stores_dict.keys():
    for zone_id, zone_data in base_zones.items():
        # PK & FK for Zones
        real_zone_id = f"{zone_id}_{store_id}"
        zones_list.append({
            'zone_id': real_zone_id,
            'store_id': store_id,
            'zone_name': zone_data['zone_name'],
            'zone_type': zone_data['zone_type'],
            'coord_lims': str(zone_data['coord_lims']) # string for the CSV
        })
        
    for camera_id, camera_data in base_cameras.items():
        # PK & FK for Cameras
        real_camera_id = f"{camera_id}_{store_id}"
        real_zone_id = f"{camera_data['zone_id']}_{store_id}"
        lims = base_zones[camera_data['zone_id']]['coord_lims']
        
        cameras_list.append({
            'camera_id': real_camera_id,
            'zone_id': real_zone_id,
            'model': camera_data['model'],
            'installation_date': fake.date_between(start_date='-2y', end_date='today').isoformat(),
            'condition': random.choice(['Optimal', 'Optimal', 'Optimal', 'Good', 'Good', 'Needs Maintenance', 'Error']),
            'lims': str(lims)
        })

df_zones = pd.DataFrame(zones_list)
df_cameras = pd.DataFrame(cameras_list)
display(df_zones.head())
display(df_cameras.head())

,zone_id,store_id,zone_name,zone_type,coord_lims
0,ENTR_SRCL,SRCL,Entrance,Entrance,None
1,WM_SRCL,SRCL,Women,Women clothes,"{'x_min': 15, 'x_max': 28, 'y_min': 3, 'y_max'..."
2,MN_SRCL,SRCL,Men,Men clothes,"{'x_min': 28, 'x_max': 40, 'y_min': 0, 'y_max'..."
3,KD_SRCL,SRCL,Kids,Kid clothes,"{'x_min': 28, 'x_max': 40, 'y_min': 8, 'y_max'..."
4,CO_SRCL,SRCL,Checkout,Checkout,"{'x_min': 15, 'x_max': 22, 'y_min': 0, 'y_max'..."


,camera_id,zone_id,model,installation_date,condition,lims
0,ENTRCM_SRCL,ENTR_SRCL,Bullet,2025-10-25,Needs Maintenance,None
1,WMCM_SRCL,WM_SRCL,Fisheye,2024-04-12,Optimal,"{'x_min': 15, 'x_max': 28, 'y_min': 3, 'y_max'..."
2,MNCM_SRCL,MN_SRCL,Fisheye,2025-09-02,Good,"{'x_min': 28, 'x_max': 40, 'y_min': 0, 'y_max'..."
3,KDCM_SRCL,KD_SRCL,Fisheye,2025-08-30,Good,"{'x_min': 28, 'x_max': 40, 'y_min': 8, 'y_max'..."
4,COCM_SRCL,CO_SRCL,Turret,2024-09-16,Needs Maintenance,"{'x_min': 15, 'x_max': 22, 'y_min': 0, 'y_max'..."


### 4. Sales & Sold Products simulation
Simulating customer checkouts. Each generated ticket maps purchased products to the specific zone of the store where the sale took place.

In [19]:
# D. SALES & E. SOLDPRODUCTS
sales_list = []
soldproducts_list = []
num_sales = 200 # number of tickets to generate

start_date = datetime(2023, 10, 1)
end_date = datetime(2023, 10, 31)

for _ in range(num_sales):
    ticket_id = f"TKT-{fake.unique.random_int(min=10000, max=99999)}"
    store_id = random.choice(list(stores_dict.keys()))
    sale_time = fake.date_time_between(start_date=start_date, end_date=end_date)
    
    # random selection of products for the ticket
    num_items = random.randint(1, 5)
    cart = random.choices(catalog, k=num_items)
    total_euros = sum(item['price'] for item in cart)
    
    sales_list.append({
        'ticket_id': ticket_id,
        'store_id': store_id,
        'timestamp': sale_time.isoformat(),
        'total_euros': round(total_euros, 2),
        'product_amount': num_items,
        'checkout_number': random.randint(1, 4)
    })
    
    for item in cart:
        soldproducts_list.append({
            'product_id': str(uuid.uuid4())[:8], # unique ID
            'ticket_id': ticket_id,
            'zone_id': f"{item['base_zone']}_{store_id}", # FK that matches the store_id
            'name': item['name'],
            'category': item['category'],
            'price': item['price']
        })

df_sales = pd.DataFrame(sales_list)
df_soldproducts = pd.DataFrame(soldproducts_list)

display(df_sales.head())
display(df_soldproducts.head())

,ticket_id,store_id,timestamp,total_euros,product_amount,checkout_number
0,TKT-42126,SRAQ,2023-10-14T05:08:11,225.99,4,3
1,TKT-62135,SRBN,2023-10-24T02:16:32,222.44,4,1
2,TKT-50352,SRCL,2023-10-15T22:00:40,31.98,2,1
3,TKT-49679,SRCL,2023-10-10T19:06:47,102.48,3,1
4,TKT-28566,SRSL,2023-10-25T04:42:19,148.00,4,2


,product_id,ticket_id,zone_id,name,category,price
0,fc22841e,TKT-42126,KD_SRAQ,Kids Winter Coat,Outerwear,45.00
1,07716a68,TKT-42126,MN_SRAQ,Men Basic T-Shirt,Tops,15.99
2,616ec10e,TKT-42126,WM_SRAQ,Women Wool Sweater,Tops,45.00
3,756f5416,TKT-42126,WM_SRAQ,Women Leather Jacket,Outerwear,120.00
4,831aa91c,TKT-62135,MN_SRBN,Men Chino Trousers,Bottoms,39.99


### 5. Store traffic & bounce rates
Simulating store traffic over a 12-hour window. The algorithm ensures the sum of people across all zones never exceeds the store's `max_capacity`.

In [20]:
# F. TRAFFIC

traffic_list = []
traffic_date = datetime(2023, 10, 1, 10, 0) # Starts at 10:00

for store_id, store_info in stores_dict.items():
    max_cap = store_info['max_capacity']
    
    # get zones for this specific store
    store_zones = [z for z in zones_list if z['store_id'] == store_id]
    
    for i in range(12): # 12 hours of operation
        current_time = traffic_date + timedelta(hours=i)
        
        # 1. random number of total visitors that never exceeds the max_capacity
        people_in_store = random.randint(5, max_cap)
        
        # 2. we start with all zones having 0 people
        zone_counts = [0] * len(store_zones)

        for _ in range(people_in_store):
            random_zone_index = random.randint(0, len(store_zones) - 1)
            zone_counts[random_zone_index] += 1

        # 3. save the records
        for zone, count in zip(store_zones, zone_counts):
            
            bounce = round(random.uniform(0.05, 0.60), 2)
            
            # peak is the actual people + a variation
            peak_variance = random.randint(0, int(count * 0.2) + 2)
            logical_peak = count + peak_variance

            traffic_list.append({
                'traffic_id': str(uuid.uuid4())[:12], # unique ID for traffic record
                'date_time': current_time.isoformat(),
                'store_id': store_id,
                'zone_id': zone['zone_id'],
                'visitor_count': count, 
                'average_time_in_store': round(random.uniform(2.0, 20.0), 1),
                'peak_people': logical_peak,
                'bounce_rate': bounce
            })

df_traffic = pd.DataFrame(traffic_list)
display(df_traffic.head())

,traffic_id,date_time,store_id,zone_id,visitor_count,average_time_in_store,peak_people,bounce_rate
0,120f325a-1a2,2023-10-01T10:00:00,SRCL,ENTR_SRCL,30,17.2,33,0.51
1,c8024239-cdc,2023-10-01T10:00:00,SRCL,WM_SRCL,51,7.8,60,0.34
2,cb167456-d5c,2023-10-01T10:00:00,SRCL,MN_SRCL,53,2.9,65,0.54
3,0233295f-969,2023-10-01T10:00:00,SRCL,KD_SRCL,49,3.0,60,0.60
4,d2414d1f-228,2023-10-01T10:00:00,SRCL,CO_SRCL,42,6.1,45,0.30


### 6. Camera detections simulation
Generating precise spatial detections. Coordinates (x, y) are randomly generated but constrained to the boundaries (`coord_lims`) of the specific camera's zone.

In [21]:
# G. DETECTIONS
detections_list = []
# 2 hour margin
det_start_time = datetime(2023, 10, 1, 12, 0)

for cam in cameras_list:
    current_time = det_start_time
    cam_lims = eval(cam['lims']) if cam['lims'] != 'None' else None
    
    for _ in range(24): # ~2 hours (1 detection every 5 mins = 12/hour)
        # between 5 & 10 minutes
        current_time += timedelta(minutes=random.randint(5, 10))
        
        # generation of a random coodinate between the limits
        coord = None
        if cam_lims:
            x = round(random.uniform(cam_lims['x_min'], cam_lims['x_max']), 2)
            y = round(random.uniform(cam_lims['y_min'], cam_lims['y_max']), 2)
            coord = f"({x}, {y})"
            
        detections_list.append({
            'detection_id': str(uuid.uuid4())[:12],
            'tracking_id': f"TRK-{random.randint(100,999)}",
            'camera_id': cam['camera_id'],
            'timestamp': current_time.isoformat(),
            'class_object': random.choice(['Client', 'Client', 'Client', 'Client', 'Staff', 'Staff']),
            'coord_lims': coord,
            'confidence': round(random.uniform(0.70, 0.99), 2)
        })

df_detections = pd.DataFrame(detections_list)
display(df_detections.head())

,detection_id,tracking_id,camera_id,timestamp,class_object,coord_lims,confidence
0,b00616cc-0ac,TRK-798,ENTRCM_SRCL,2023-10-01T12:05:00,Staff,None,0.82
1,984e2a53-e35,TRK-192,ENTRCM_SRCL,2023-10-01T12:12:00,Staff,None,0.80
2,4357f5a9-c1f,TRK-143,ENTRCM_SRCL,2023-10-01T12:18:00,Staff,None,0.80
3,38219945-4ae,TRK-291,ENTRCM_SRCL,2023-10-01T12:27:00,Client,None,0.87
4,9b66f694-ffe,TRK-300,ENTRCM_SRCL,2023-10-01T12:35:00,Client,None,0.78


### 7. Data export (CSV)

    Warning: The data was already generated, as you can see in the data directory. If you want to remake the data, you can run this line of code. But as we want to all have the same data, don't run it (As you will modify the Git csv files).

Finally, we create a `./data` directory (if it doesn't exist) and save all our generated dataframes to CSV files for further analysis.

In [27]:
# 3. export to csv

"""
df_stores.to_csv('./data/stores.csv', index=False)
df_zones.to_csv('./data/zones.csv', index=False)
df_cameras.to_csv('./data/cameras.csv', index=False)
df_sales.to_csv('./data/sales.csv', index=False)
df_soldproducts.to_csv('./data/soldproducts.csv', index=False)
df_traffic.to_csv('./data/traffic.csv', index=False)
df_detections.to_csv('./data/detections.csv', index=False)
"""

print("CSV files succesfully created.")

CSV files succesfully created.


---

### 8. Saving data to Data Base - **ONLY RAW DATA** - Developer
We can store the raw generated data into the Data Base for testing. We manage all the data from the files. After running the script, the clean data will be stored into the Data Base. This is used to only test the part of your script. **Be aware if you use this code...**

In [29]:
"""
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
DATABASE_URL = os.getenv('CHURRO')
engine = create_engine(DATABASE_URL)

df_stores = pd.read_csv('./data/stores.csv')
df_sales = pd.read_csv('./data/sales.csv')
df_zones = pd.read_csv('./data/zones.csv')
df_traffic = pd.read_csv('./data/traffic.csv')
df_cameras = pd.read_csv('./data/cameras.csv')
df_detections = pd.read_csv('./data/detections.csv')
df_soldproducts = pd.read_csv('./data/soldproducts.csv')

df_stores.to_sql('stores', con=engine, if_exists='append', index=False)
df_sales.to_sql('sales', con=engine, if_exists='append', index=False)
df_zones.to_sql('zones', con=engine, if_exists='append', index=False)
df_traffic.to_sql('traffic', con=engine, if_exists='append', index=False)
df_cameras.to_sql('camera', con=engine, if_exists='append', index=False)
df_detections.to_sql('detection', con=engine, if_exists='append', index=False)
df_soldproducts.to_sql('product', con=engine, if_exists='append', index=False)
"""

"\nimport os\nfrom sqlalchemy import create_engine\nfrom dotenv import load_dotenv\n\nload_dotenv()\nDATABASE_URL = os.getenv('CHURRO')\nengine = create_engine(DATABASE_URL)\n\ndf_stores = pd.read_csv('./data/stores.csv')\ndf_sales = pd.read_csv('./data/sales.csv')\ndf_zones = pd.read_csv('./data/zones.csv')\ndf_traffic = pd.read_csv('./data/traffic.csv')\ndf_cameras = pd.read_csv('./data/cameras.csv')\ndf_detections = pd.read_csv('./data/detections.csv')\ndf_soldproducts = pd.read_csv('./data/soldproducts.csv')\n\ndf_stores.to_sql('stores', con=engine, if_exists='append', index=False)\ndf_sales.to_sql('sales', con=engine, if_exists='append', index=False)\ndf_zones.to_sql('zones', con=engine, if_exists='append', index=False)\ndf_traffic.to_sql('traffic', con=engine, if_exists='append', index=False)\ndf_cameras.to_sql('camera', con=engine, if_exists='append', index=False)\ndf_detections.to_sql('detection', con=engine, if_exists='append', index=False)\ndf_soldproducts.to_sql('product', c